# CyberShield — Cyberbullying Detection (NLP Multi-Label Classification)

**AI Assignment (Title 4: Natural Language Processing)**

Run the cells **top to bottom** (Shift+Enter, or menu **Run → Run All Cells**).

| Member | Method |
|--------|--------|
| Member 1 | Logistic Regression + TF-IDF |
| Member 2 | Linear SVM + TF-IDF |
| Member 3 | Random Forest + TF-IDF |

This notebook trains and evaluates the models. To use the **web app**, run
`streamlit run app.py` in a terminal afterwards.

## Running in Google Colab?

On your own computer (Anaconda/Jupyter), **skip this** — go to "0. Setup".

In **Google Colab**: run the cell below, then upload `cyberbully_detection.zip`
when prompted. Once per session.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os, zipfile
    from google.colab import files

    if not os.path.isdir("cyberbully_detection"):
        print("Please choose cyberbully_detection.zip in the upload box below...")
        uploaded = files.upload()
        zip_name = [f for f in uploaded if f.endswith(".zip")][0]
        with zipfile.ZipFile(zip_name) as z:
            z.extractall(".")
        print("Extracted.")

    os.chdir("cyberbully_detection")
    print("Now working in:", os.getcwd())

    # (Optional) keep your results after the Colab session ends by saving to
    # Google Drive instead. Uncomment the 3 lines below if you want this:
    # from google.colab import drive
    # drive.mount("/content/drive")
    # os.chdir("/content/drive/MyDrive/cyberbully_detection")

    print("Installing/upgrading a couple of packages Colab may be missing...")
    !pip install -q streamlit joblib
else:
    print("Not running in Colab - skipping upload step.")


## 0. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
%matplotlib inline

print("Working directory:", os.getcwd())

# This notebook must run from the project root (the folder containing 'src', 'models', 'data').
# If the check below fails, set the correct path in os.chdir() and re-run this cell:
# os.chdir(r"C:\Users\YourName\Desktop\cyberbully_detection")

assert os.path.isdir("src") and os.path.isdir("data"), (
    "Not in the project root. Open the notebook from the cyberbully_detection folder, "
    "or set the path with os.chdir() above."
)
print("Setup OK - project root found.")

## 1. Load data and preprocess text
This loads the HateXplain dataset, builds the 6 binary labels, cleans the text, and makes the train/test split. All three models share this same split so the comparison is fair.

In [ ]:
from src.common import prepare_data

X_train, X_test, y_train, y_test, labels = prepare_data(data_dir="data")
print("\nLabels:", labels)
print("Train comments:", len(X_train), "| Test comments:", len(X_test))

## 2. Explore the data (EDA)
Charts for your documentation's *Methodology / Dataset* section.

In [ ]:
import matplotlib.pyplot as plt
from src.data_loader import load_dataset

df_full, text_col, label_cols = load_dataset("data", verbose=False)

# (a) how many comments carry each label
counts = df_full[label_cols].sum().sort_values(ascending=False)
counts.plot(kind="bar", color="#c0392b", figsize=(8,4),
            title="Number of comments per label")
plt.ylabel("Count"); plt.tight_layout(); plt.show()

# (b) comment length distribution
lengths = df_full[text_col].astype(str).str.split().apply(len)
plt.figure(figsize=(8,4))
plt.hist(lengths, bins=50, color="#2980b9")
plt.title("Comment length (words)"); plt.xlabel("Words per comment")
plt.xlim(0, lengths.quantile(0.99)); plt.tight_layout(); plt.show()

print("Total comments:", len(df_full))

## 3. Member 1 — Logistic Regression
Linear model on TF-IDF features, one classifier per label.

In [ ]:
import joblib, os
from models.member1_logistic_regression import build_pipeline as build_lr
from src.evaluate import evaluate_model, save_result

lr = build_lr()
lr.fit(X_train, y_train)
lr_scores = evaluate_model("Logistic Regression", y_test.values, lr.predict(X_test), labels)
save_result(lr_scores)
os.makedirs("results", exist_ok=True)
joblib.dump({"pipeline": lr, "labels": labels}, "results/model_lr.joblib", compress=3)
print("Saved -> results/model_lr.joblib")

## 4. Member 2 — Linear SVM
Maximum-margin linear classifier on the same TF-IDF features.

In [ ]:
from models.member2_svm import build_pipeline as build_svm

svm = build_svm()
svm.fit(X_train, y_train)
svm_scores = evaluate_model("Linear SVM", y_test.values, svm.predict(X_test), labels)
save_result(svm_scores)
joblib.dump({"pipeline": svm, "labels": labels}, "results/model_svm.joblib", compress=3)
print("Saved -> results/model_svm.joblib")

## 5. Member 3 — Random Forest
An ensemble of decision trees. Slower to train (~1 minute) and captures non-linear word combinations.

In [ ]:
from models.member3_random_forest import build_pipeline as build_rf

rf = build_rf()
rf.fit(X_train, y_train)
rf_scores = evaluate_model("Random Forest", y_test.values, rf.predict(X_test), labels)
save_result(rf_scores)
joblib.dump({"pipeline": rf, "labels": labels}, "results/model_rf.joblib", compress=3)
print("Saved -> results/model_rf.joblib")

## 6. Optional: Naive Bayes (4th model, for testing)

**Not one of the assignment's three required methods** (those are Logistic
Regression, SVM, and Random Forest above) — included here so you can compare
a probabilistic model against the required three before deciding whether to
use it. See `models/naive_bayes_extra.py` for the full write-up of why it was
added: it's specifically well-suited to word/n-gram count data like TF-IDF,
with no ensemble-voting dilution the way Random Forest has on sparse text.

Trade-off to know about (also documented in the model file): Naive Bayes had
**higher** recall on the core "abusive" flag than Random Forest in testing,
but **lower** recall on the minority target categories (Gender, Sexual
Orientation, Miscellaneous). Its confidence scores also tend to run to
extremes (near 0% or 100%) due to its feature-independence assumption -
don't read them as precisely calibrated probabilities.

In [ ]:
from models.naive_bayes_extra import build_pipeline as build_nb

nb_model = build_nb()
nb_model.fit(X_train, y_train)
nb_scores = evaluate_model("Naive Bayes", y_test.values, nb_model.predict(X_test), labels)
save_result(nb_scores)
joblib.dump({"pipeline": nb_model, "labels": labels}, "results/model_nb.joblib", compress=3)
print("Saved -> results/model_nb.joblib")

## 7. Compare all models
The comparison table + chart for your *Results & Discussion* section.

In [ ]:
import pandas as pd

scores = pd.read_csv("results/model_scores.csv")
cols = ["model","accuracy","subset_accuracy","hamming_loss","f1_micro","f1_macro",
        "precision_micro","recall_micro","train_time_sec","predict_time_sec"]
# "accuracy" = per-label accuracy averaged across categories - the number
# comparable to what most classification papers report. "subset_accuracy" is
# the stricter "all 6 labels correct at once" measure and will look lower -
# that is expected for multi-label problems, not a sign of a weak model.
scores = scores[[c for c in cols if c in scores.columns]]
from IPython.display import display
display(scores)

x = range(len(scores)); w = 0.35
plt.figure(figsize=(9,5))
plt.bar([i-w/2 for i in x], scores["f1_micro"], w, label="F1 (micro)", color="#c0392b")
plt.bar([i+w/2 for i in x], scores["f1_macro"], w, label="F1 (macro)", color="#2980b9")
plt.xticks(list(x), scores["model"], rotation=15, ha="right")
plt.ylim(0,1); plt.ylabel("F1 score"); plt.title("Model comparison"); plt.legend()
plt.tight_layout(); plt.savefig("results/comparison_f1.png", dpi=120); plt.show()

best = scores.loc[scores["f1_micro"].idxmax(), "model"]
print("Best model by micro-F1:", best)

## 8. Find each model's own best detection threshold

Different algorithms produce confidence scores on different natural scales,
even when equally correct - sharing one threshold across all of them
quietly penalizes some far more than others (Random Forest was hit hardest
in testing: F1 dropped from 0.70 at its own best threshold to 0.56 when
forced to share a threshold tuned for the linear models). This re-derives
each model's own F1-optimal threshold directly from the models you just
trained - re-run this any time you retrain, since the right value is
specific to that exact trained model, not the algorithm in general.

In [ ]:
from src.threshold_sweep import sweep

best_thresholds = sweep()
print()
print("Copy the printed DEFAULT_THRESHOLDS block above into src/config.py")
print("if any of these values changed from what's currently there.")

## 9. Try it on your own comment
Type any sentence and see what the model predicts.

In [ ]:
from src.predictor import predict as model_predict
from src.config import DEFAULT_THRESHOLDS

def predict_comment(text, model_path="results/model_lr.joblib", model_name="Logistic Regression"):
    bundle = joblib.load(model_path)
    threshold = DEFAULT_THRESHOLDS.get(model_name, 0.5)   # each model's own verified best threshold
    result = model_predict(bundle, text, threshold=threshold)
    print("Comment  :", text)
    print("Model    :", model_name, f"(threshold={threshold})")
    print("Verdict  :", ("CYBERBULLYING -> " + ", ".join(result["flagged"])) if result["is_bully"] else "Clean")
    print("Confidence per category:", {k: round(v, 3) for k, v in result["probs"].items()})
    return result

predict_comment("you are a stupid idiot, nobody likes you")

# Try other models by changing model_path/model_name, e.g.:
# predict_comment("you are a stupid idiot, nobody likes you",
#                  model_path="results/model_rf.joblib", model_name="Random Forest")